# Phrase Reduction Pipeline (BeauVis-Inspired, v2)

Revised pipeline addressing issues from the original BeauVisPhraseReduction notebook:
- **No fixed shortlist cap** — adaptive sizing so every topic (including Schema) gets sub-topic phrases
- **Criterion 5 revised** — matches BeauVis's "clearly applies to a visual representation" instead of excluding single-VisType terms
- **`secondary_topic` column** added to output
- **Stops at Step 5** for review before further processing

### Pipeline Steps
1. **Load data** — Same Google Sheets source
2. **Merge-first consolidation** — Manual synonym groups + TF-IDF cosine similarity for rare keywords
3. **Criterion-based filtering** — Revised criteria (no single-VisType exclusion)
4. **Universality scoring** — score = count × n_vistypes, with primary & secondary topic assignment
5. **Adaptive topic-balanced selection** — Ensures all 7 topics have minimum representation
6. **Save shortlist + tracking table** — Output to `phrase_reduction_v2/`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

VisTypes = ['Area','Bar','Cont.-ColorPatn','Glyph','Grid','Line','Node-link','Point','Text']

topic_names = [
    'Data Density / Image Clutter',
    'Visual Encoding Clarity',
    'Semantics / Text Legibility',
    'Schema',
    'Color, Symbol, and Texture Details',
    'Aesthetics Uncertainty',
    'Immediacy / Cognitive Load'
]

topic_abbrev = {
    'Data Density / Image Clutter': 'DDIC',
    'Visual Encoding Clarity': 'VEC',
    'Semantics / Text Legibility': 'STL',
    'Schema': 'SCH',
    'Color, Symbol, and Texture Details': 'CSTD',
    'Aesthetics Uncertainty': 'AU',
    'Immediacy / Cognitive Load': 'ICL'
}

topic_colors = {
    'Data Density / Image Clutter': '#1f77b4',
    'Visual Encoding Clarity': '#ff7f0e',
    'Semantics / Text Legibility': '#2ca02c',
    'Schema': '#d62728',
    'Color, Symbol, and Texture Details': '#9467bd',
    'Aesthetics Uncertainty': '#8c564b',
    'Immediacy / Cognitive Load': '#e377c2'
}

TOPIC_SEP = ' | '

# Output sub-folder
out_dir = os.path.join(os.getcwd(), 'phrase_reduction_v2')
os.makedirs(out_dir, exist_ok=True)
print(f'Output directory: {out_dir}')

Output directory: d:\Coding\Copilot\comment_post_processing\phrase_reduction_v2


## Step 0: Load Data

In [2]:
# Load keyword counts from Google Sheets pivot table
spreadsheet_id = '1cVCOfBjHmcvsyrn7U0CbXoBSksF4JDRqWYm-ZJPbL4k'
sheet_id = '517808299'
df_keyword_counts = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?gid={sheet_id}&format=csv'
)
# Source data renamed 'keyword' → 'phrase'; normalise to 'keyword' for downstream code
if 'phrase' in df_keyword_counts.columns and 'keyword' not in df_keyword_counts.columns:
    df_keyword_counts = df_keyword_counts.rename(columns={'phrase': 'keyword'})
print(f'Raw records: {len(df_keyword_counts)}')

# Build unique phrases table
vt_involvement = (
    df_keyword_counts
    .groupby('keyword')[VisTypes]
    .apply(lambda g: ','.join([vt for vt in VisTypes if (g[vt] > 0).any()]))
    .reset_index()
    .rename(columns={0: 'vistypes_involved'})
)
vt_involvement['n_vistypes'] = vt_involvement['vistypes_involved'].apply(
    lambda x: len(x.split(',')) if x else 0
)

topic_involvement = (
    df_keyword_counts
    .groupby('keyword')['topic']
    .apply(lambda g: TOPIC_SEP.join(sorted(g.unique())))
    .reset_index()
    .rename(columns={'topic': 'topics_involved'})
)

kw_counts = (
    df_keyword_counts
    .groupby('keyword', as_index=False)['num_images']
    .sum()
    .rename(columns={'num_images': 'count'})
)

df_all = (kw_counts
    .merge(vt_involvement, on='keyword')
    .merge(topic_involvement, on='keyword')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

print(f'Unique phrases: {len(df_all)}')
df_all.head(10)

Raw records: 404
Unique phrases: 399


,keyword,count,vistypes_involved,n_vistypes,topics_involved
0,color variety/arrangement/distribution,87,"Area,Bar,Cont.-ColorPatn,Glyph,Grid,Line,Node-...",9,"Color, Symbol, and Texture Details"
1,more charts/points/lines/shapes/elements,79,"Area,Bar,Cont.-ColorPatn,Glyph,Grid,Line,Node-...",9,Data Density / Image Clutter
2,much/more data/info/info spread,65,"Area,Bar,Cont.-ColorPatn,Glyph,Grid,Line,Node-...",9,Data Density / Image Clutter
3,easy to interpret/read/understand,55,"Area,Bar,Cont.-ColorPatn,Glyph,Grid,Line,Node-...",9,Immediacy / Cognitive Load
4,"domain-specific concepts (e.g., chemical, biol...",41,"Area,Cont.-ColorPatn,Glyph,Grid,Line,Node-link...",8,Schema
5,more detailed/things,37,"Area,Bar,Cont.-ColorPatn,Glyph,Grid,Node-link,...",8,Data Density / Image Clutter
6,title/axis/label/descriptions,37,"Area,Bar,Cont.-ColorPatn,Glyph,Grid,Line,Node-...",9,Semantics / Text Legibility
7,unclear colormaps,34,"Area,Bar,Cont.-ColorPatn,Glyph,Grid,Line,Node-...",9,"Color, Symbol, and Texture Details"
8,lack of/not enough axis labels/legend/annotati...,34,"Area,Bar,Cont.-ColorPatn,Glyph,Line,Node-link,...",8,Semantics / Text Legibility
9,hard to interpret/read/understand,33,"Area,Bar,Cont.-ColorPatn,Glyph,Grid,Line,Node-...",9,Immediacy / Cognitive Load


In [3]:
# Explore count distribution and topic coverage
rare = df_all[df_all['count'] < 3]
survivors = df_all[df_all['count'] >= 3]
print(f'Phrases with count < 3 (rare):  {len(rare)}')
print(f'Phrases with count >= 3:        {len(survivors)}')
print(f'\nCount distribution:')
print(df_all['count'].value_counts().sort_index().head(10).to_string())

# Per-topic phrase counts (based on primary topic = first listed)
print(f'\n--- Phrases per topic (primary assignment) ---')
for topic in topic_names:
    abbrev = topic_abbrev[topic]
    mask = df_all['topics_involved'].str.contains(re.escape(topic), na=False)
    n_total = mask.sum()
    n_rare = (mask & (df_all['count'] < 3)).sum()
    n_surv = (mask & (df_all['count'] >= 3)).sum()
    print(f'  [{abbrev:4s}] {topic:45s}: {n_total:3d} total ({n_surv:3d} count>=3, {n_rare:3d} rare)')

Phrases with count < 3 (rare):  251
Phrases with count >= 3:        148

Count distribution:
count
1     185
2      66
3      36
4      16
5      17
6      10
7      11
8       5
9       9
10      7

--- Phrases per topic (primary assignment) ---
  [DDIC] Data Density / Image Clutter                 :  41 total ( 23 count>=3,  18 rare)
  [VEC ] Visual Encoding Clarity                      : 129 total ( 36 count>=3,  93 rare)
  [STL ] Semantics / Text Legibility                  :  84 total ( 21 count>=3,  63 rare)
  [SCH ] Schema                                       :  12 total (  9 count>=3,   3 rare)
  [CSTD] Color, Symbol, and Texture Details           :  73 total ( 28 count>=3,  45 rare)
  [AU  ] Aesthetics Uncertainty                       :  14 total (  5 count>=3,   9 rare)
  [ICL ] Immediacy / Cognitive Load                   :  51 total ( 31 count>=3,  20 rare)


## Step 1: Merge-First Consolidation (Per-Topic TF-IDF)

Instead of discarding ~275 rare phrases (count ≤ 2), merge each into its closest higher-count match:

1. **Manual synonym groups** — Hand-curated groups of near-synonyms (rare + frequent), already organized per-topic
2. **TF-IDF cosine similarity (per-topic)** — For remaining rare phrases, auto-merge within the **same primary topic** only (avoids cross-topic mismerges). Falls back to global pool only if a topic has zero survivors.
3. **Truly unmatched** — Phrases with no good match are force-merged into closest same-topic survivor

In [4]:
# ---- Step 1: Unified Merge-First Consolidation ----
MIN_COUNT = 3
SIMILARITY_THRESHOLD = 0.35

df_rare = df_all[df_all['count'] < MIN_COUNT].copy()
df_survivors = df_all[df_all['count'] >= MIN_COUNT].copy()

print(f'Starting: {len(df_all)} phrases total')
print(f'  Rare (count < {MIN_COUNT}): {len(df_rare)}')
print(f'  Survivors (count >= {MIN_COUNT}): {len(df_survivors)}')

# ---- Helper: primary topic (first listed) ----
def get_primary_topic_str(topics_str):
    """Return first topic from a TOPIC_SEP-joined string."""
    if pd.isna(topics_str) or not topics_str:
        return 'Unknown'
    return topics_str.split(TOPIC_SEP)[0].strip()

df_all['_primary_topic'] = df_all['topics_involved'].apply(get_primary_topic_str)
df_rare['_primary_topic'] = df_rare['topics_involved'].apply(get_primary_topic_str)
df_survivors['_primary_topic'] = df_survivors['topics_involved'].apply(get_primary_topic_str)

# ---- Manual synonym groups (unchanged — already per-topic by design) ----
synonym_groups = [
    # --- Immediacy / Cognitive Load (polarity-merged: hard + easy) ---
    ['more difficult to interpret/read/understand/differentiate',
     'not interpretable/readable/understandable',
     'hard to interpret/read/understand/differentiate',
     'hard to interpret/read/understand',
     'hard to compare/discern/distinguish',
     'hard to see/tell/visualize',
     'hard to focus/follow',
     'hard to identify/extract/find features',
     'more complicated',
     'more difficult',
     'complicated to process',
     'hard to read', 'hard to describe', 'hard to measure/model',
     'hard to read shape', 'hard to distinguish colors',
     'harder to view data point', 'struggled to read',
     'less understandable', 'not easily understood', 'less clear',
     'less intuitive', 'low readability', 'unreadable',
     'nothing clear', 'less simple', 'more intricate',
     'easy to interpret/read/understand',
     'easier to interpret/read/understand',
     'easier to process/visualize',
     'simple to interpret/read/understand',
     'easier to see/tell/visualize',
     'easy to focus/follow',
     'easy to understand',
     'intuitive',
     'easy to read', 'easy to derive meaning',
     'easy to identify/extract/find features',
     'easier to compare/discern/distinguish',
     'fast readability', 'more interpretable/readable/understandable',
     'more legible', 'description easy to read',
     'more difficult to compare/discern/distinguish'],

    ['take longer to interpret',
     'more reading/interpretation/understanding',
     'more effort/reading/detailed analysis',
     'attention/squinting to understand',
     'more to read/analyze/understand',
     'more thinking', 'multiple interpretation',
     'provokes thought and understanding',
     'require specialized knowledge',
     'not enough knowledge to tell'],

    # --- Data Density / Image Clutter ---
    ['more charts/points/lines/shapes/elements',
     'multiple features/elements/graphs',
     'many points/lines/shapes/elements',
     'too many details/divisions',
     'many/more data/info',
     'few points/lines/shapes/elements',
     'dense points/lines/shapes/elements',
     'too many points/lines/shapes/elements',
     'too more points/lines/shapes/elements',
     'multiple points/lines/shapes/elements',
     'multiple shapes', 'multiple interacting elements',
     'multiple aspects', 'more forms',
     'too many sections', 'too many subjects',
     'too many subplots', 'more pixels'],

    ['much/more data/info/info spread',
     'more detailed/things',
     'too much data/info',
     'fine/layered details',
     'large dataset', 'diverse information',
     'more nuance information',
     'simple information', 'no data', 'no detail',
     'less detailed/things', 'no more information',
     'multi-year information', 'many measurements',
     'mixed timeline', 'single variable',
     'little data/info'],

    ['dense/cluttered data/info',
     'dense/cluttered layout',
     'messy/mixed up/noisy/intermingled elements',
     'overlapping shapes/colors/lines',
     'dense/cluttered shape', 'cluttered labels/annotations',
     'concentrated', 'dots scattered', 'scattered',
     'scattered squares', 'less empty space',
     'less negative space', 'negative space',
     'convoluted composition', 'information overload',
     'complex design'],

    # --- Color, Symbol, and Texture Details (merged: variety+shading, contrast+coloring) ---
    ['color variety/arrangement/distribution',
     'too many colors',
     'amount of / too many colors',
     'coloring',
     'different colors', 'more colors', 'bold colors',
     'appealing colors', 'intense/too many colors',
     'salient colors', 'similar colors', 'same colors',
     'not simple color-wise', 'very different colors',
     'colored bars/charts', 'colored surface',
     'colorless', 'full color image', 'color represents sounds',
     'color shading', 'black-to-white/color gradient',
     'color scale/scheme',
     'color hues', 'color intensity', 'color saturation',
     'color details', 'color dimension', 'color superposition/layer',
     'blended colors', 'clearer coloring',
     'color keys', 'color represents groups',
     'separate color scales', 'color improves readability',
     'white color on black background', 'black color',
     'blue color', 'green color'],

    ['lack of color contrast / hard to distinguish colors',
     'unclear color meaning',
     'ambiguous/confusing colors',
     'unclear colormaps',
     'more contrast',
     'high contrast',
     'good color contrast/separation/segmentation',
     'low color contrast/separation/segmentation',
     'no color difference', 'color differentiation',
     'separate values by color to show differences',
     'shapes and contrasts',
     'unclear coloring'],

    ['symbols',
     'texture details',
     'striped texture', 'indicating symbols',
     'visually stimulating  texture/colors',
     'icons', 'nodes', 'symbols and shading'],

    # --- Semantics / Text Legibility ---
    ['amount of words/context/numbers',
     'more texts/words',
     'word/text/sentence',
     'too many texts/words', 'texts/words', 'text',
     'text ratio', 'code/text', 'context/text/description',
     'too many numbers', 'many numbers', 'more numbers',
     'ambiguous numbers', 'words and numbers',
     'numbers and shapes', 'involves numbers',
     'word frequency', 'word source', 'more text boxes',
     'text on picture', 'text and imagery'],

    # (merged: labels + legends + measurement)
    ['title/axis/label',
     'more title/axis/label',
     'clear title/axis/label',
     'lack of/not enough axis labels/legend/annotations/context',
     'title/axis/label/descriptions',
     'unclear labels', 'different labels',
     'labeled axis', 'axis meaning', 'axis numbers',
     'captions', 'highlighted words',
     'label meaning', 'almost no explanation',
     'axis scales', 'x-axis unlabeled',
     'too much legend', 'legend easy to understand',
     'lack of legend', 'vague legend',
     'explanatory legend', 'legend on the top',
     'with legend', 'no interaction',
     'many legend categories', 'number of legend items',
     'no measurement/metric/axis'],

    ['word rotation/small font size',
     'different font/word sizes/structure',
     'unclear writing',
     'readability',
     'easier to read',
     'text hard to read', 'font', 'single font',
     'small', 'small and simple',
     'hierarchical text', 'order of words',
     'explanation text', 'detailed inscription',
     'description', 'word misoriented'],

    # --- Visual Encoding Clarity ---
    # (merged: shapes + lines)
    ['curves and shapes',
     'shape variety',
     'shape size variation',
     'element sizes',
     'shapes', 'shapes and lines',
     'shape', 'overall shape', 'clear shape',
     'unclear shape', 'shape arrangement', 'shape indicators',
     'shape misunderstanding', 'shape subset representation',
     'shape-color mix', 'shapes varied',
     'similar shapes', 'less organic shapes',
     'less uniform shapes', 'uneven shape',
     'categorized shapes', 'colored shape',
     'similar box sizes', 'mixed size', 'size',
     'size variation', 'size decreasing',
     'more squares than shapes',
     'image size', 'length', 'length variation', 'squares',
     'straight lines', 'curved lines',
     'indistinguishable/unintuitive lines', 'lines',
     'few lines', 'smooth lines', 'intersecting lines',
     'precision of lines', 'line movement',
     'line thickness variation', 'line quantity representation',
     'bars as individual lines', 'horizontal bars',
     'number of vertical lines', 'lines and scales',
     'error bars'],

    ['clearer indication',
     'clear indication',
     'unclear shape meaning',
     'unclear meaning',
     'no clear indication',
     'unclear meaning/confusing',
     'indication', 'unclear structure',
     'unclear what data conveys', 'unclear where to look/what to see',
     'unclear relation between words', 'unclear filled bars',
     'lack of meaning', 'different word meaning',
     'code meaning', 'visualization meaning',
     'different clarity',
     'unclear movement', 'confusing connection', 'visual cues'],

    ['2D/3D',
     'organized/structured',
     'spatial organization', 'grid layout',
     'image layout', 'disposition',
     'more uniform design', 'misaligned',
     'value and graphic alignment',
     'number of levels', 'number distribution',
     'data intersection', 'data trend',
     'image location', 'image portrayal',
     'value representation',
     'frame with values', 'values in the middle',
     'more axes', 'fewer axes'],

    # --- Schema ---
    ['domain-specific concepts (e.g., chemical, biology, map)',
     'unfamiliar concepts/patterns',
     'a specific technique',
     'a specific technique, e.g., bar, pie, circle',
     'a specific domain',
     'abstract',
     'no context or reference',
     'familiar representation',
     'technical encoding', 'network representation',
     'metric representation', 'unknown hidden content',
     'multivariate', 'unique', 'unique data point',
     'nonlinearity', 'nonlinear measurement/metric/axis',
     'recognizable objects', 'graph', 'plots',
     'infomration representation', 'representation',
     'context', 'field information',
     'arbitrary imaging', 'drawing style',
     'graphical information', 'factual information',
     'information nuance expressed'],

    # --- Aesthetics Uncertainty (polarity-merged: distracting + clarity) ---
    ['distracting/confusing/unclear',
     'looks random/messy/lack structure',
     'less attractive',
     'feeling strange', 'visually overwhelming',
     'visually striking', 'stimulus less visible',
     'nothing visible', 'inconsistent', 'irregular',
     'random-like movement', 'unable to sort',
     'pixeled picture', 'different movement',
     'visual clarity/appealing',
     'not intuitive/simple data',
     'clear and concise', 'clear connection',
     'clear delineation'],
]

# ---- Build phrase→representative mapping from manual groups ----
all_kws_set = set(df_all['keyword'])
manual_merge_map = {}

for group in synonym_groups:
    members_in_data = [kw for kw in group if kw in all_kws_set]
    if len(members_in_data) <= 1:
        continue
    counts = {kw: df_all.loc[df_all['keyword'] == kw, 'count'].values[0] for kw in members_in_data}
    representative = max(counts, key=counts.get)
    for kw in members_in_data:
        manual_merge_map[kw] = representative

n_manual_rare = sum(1 for kw in df_rare['keyword'] if kw in manual_merge_map and manual_merge_map[kw] != kw)
print(f'\nManual synonym groups: {len(synonym_groups)}')
print(f'Rare phrases matched by manual groups: {n_manual_rare}')

# ---- TF-IDF matching: PER-TOPIC for remaining rare phrases ----
rare_group_reps = set()
for kw in df_rare['keyword']:
    if kw in manual_merge_map and manual_merge_map[kw] == kw:
        rare_group_reps.add(kw)

rare_unmatched = [kw for kw in df_rare['keyword']
                  if kw not in manual_merge_map and kw not in rare_group_reps]

# Build topic lookup for rare_unmatched
rare_topic_lookup = dict(zip(df_rare['keyword'], df_rare['_primary_topic']))
surv_topic_lookup = dict(zip(df_survivors['keyword'], df_survivors['_primary_topic']))

print(f'Rare phrases needing TF-IDF matching: {len(rare_unmatched)}')

def preprocess_kw(kw):
    """Expand slashes and clean punctuation for TF-IDF."""
    kw = kw.lower()
    kw = kw.replace('/', ' ')
    kw = kw.replace('-', ' ')
    kw = re.sub(r'[^a-z0-9\s]', ' ', kw)
    return ' '.join(kw.split())

# Group rare unmatched and survivors by primary topic
survivor_kws_all = df_survivors['keyword'].tolist()

tfidf_merge_map = {}
tfidf_force_merged = []

print(f'\n--- TF-IDF per-topic matching ---')
for topic in topic_names:
    abbrev = topic_abbrev[topic]
    # Survivors in this topic
    topic_surv_kws = [kw for kw in survivor_kws_all if surv_topic_lookup.get(kw) == topic]
    # Rare unmatched in this topic
    topic_rare_kws = [kw for kw in rare_unmatched if rare_topic_lookup.get(kw) == topic]

    if not topic_rare_kws:
        print(f'  [{abbrev:4s}] 0 rare to match')
        continue

    if not topic_surv_kws:
        # No same-topic survivors — fall back to global survivors
        print(f'  [{abbrev:4s}] {len(topic_rare_kws)} rare, 0 same-topic survivors → global fallback')
        topic_surv_kws = survivor_kws_all

    # Build TF-IDF within this topic pool
    all_texts = ([preprocess_kw(kw) for kw in topic_surv_kws]
                 + [preprocess_kw(kw) for kw in topic_rare_kws])
    vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1)
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    n_s = len(topic_surv_kws)
    surv_matrix = tfidf_matrix[:n_s]
    rare_matrix = tfidf_matrix[n_s:]
    sim_matrix = cosine_similarity(rare_matrix, surv_matrix)

    n_auto = 0
    n_force = 0
    for i, rare_kw in enumerate(topic_rare_kws):
        best_idx = sim_matrix[i].argmax()
        best_sim = sim_matrix[i, best_idx]
        best_match = topic_surv_kws[best_idx]
        tfidf_merge_map[rare_kw] = (best_match, best_sim)
        if best_sim < SIMILARITY_THRESHOLD:
            tfidf_force_merged.append((rare_kw, best_match, best_sim))
            n_force += 1
        else:
            n_auto += 1

    print(f'  [{abbrev:4s}] {len(topic_rare_kws)} rare → matched within {len(topic_surv_kws)} '
          f'survivors (auto={n_auto}, force={n_force})')

print(f'\nTF-IDF total auto-merged (sim >= {SIMILARITY_THRESHOLD}): '
      f'{len(tfidf_merge_map) - len(tfidf_force_merged)}')
print(f'TF-IDF total force-merged (sim < {SIMILARITY_THRESHOLD}): {len(tfidf_force_merged)}')
print(f'  (No phrases discarded — all rare merged into closest same-topic synonym)')

# ---- Combine all merges ----
merge_map = {}

for kw, rep in manual_merge_map.items():
    if kw != rep:
        merge_map[kw] = rep

for kw, (target, sim) in tfidf_merge_map.items():
    final_target = manual_merge_map.get(target, target)
    merge_map[kw] = final_target

# ---- Apply merges: build consolidated DataFrame ----
all_representatives = set(df_all['keyword']) - set(merge_map.keys())

df_step1 = df_all[df_all['keyword'].isin(all_representatives)].copy()

for _, row in df_all.iterrows():
    kw = row['keyword']
    if kw in merge_map:
        rep = merge_map[kw]
        mask = df_step1['keyword'] == rep
        if mask.any():
            df_step1.loc[mask, 'count'] += row['count']
            existing_vt = set(df_step1.loc[mask, 'vistypes_involved'].values[0].split(','))
            new_vt = set(row['vistypes_involved'].split(',')) if row['vistypes_involved'] else set()
            merged_vt = ','.join(sorted((existing_vt | new_vt) & set(VisTypes)))
            df_step1.loc[mask, 'vistypes_involved'] = merged_vt
            df_step1.loc[mask, 'n_vistypes'] = len(merged_vt.split(',')) if merged_vt else 0
            existing_topics = set(df_step1.loc[mask, 'topics_involved'].values[0].split(TOPIC_SEP))
            new_topics = set(row['topics_involved'].split(TOPIC_SEP)) if row['topics_involved'] else set()
            merged_topics = TOPIC_SEP.join(sorted((existing_topics | new_topics) & set(topic_names)))
            df_step1.loc[mask, 'topics_involved'] = merged_topics

df_step1 = df_step1.sort_values('count', ascending=False).reset_index(drop=True)

# ---- Rename representative phrases to generic names ----
phrase_rename_map = {
    'easy to interpret/read/understand': 'easy/hard to interpret',
    'much/more data/info/info spread': 'much/little data/info',
    'title/axis/label': 'labels/axes/legends',
    'title/axis/label/descriptions': 'labels/axes/legends',
    'color variety/arrangement/distribution': 'color variety/shading',
    'lack of color contrast / hard to distinguish colors': 'color contrast/clarity',
    'unclear colormaps': 'color contrast/clarity',
}
df_step1['keyword'] = df_step1['keyword'].replace(phrase_rename_map)
for kw in list(merge_map.keys()):
    if merge_map[kw] in phrase_rename_map:
        merge_map[kw] = phrase_rename_map[merge_map[kw]]
for kw in list(manual_merge_map.keys()):
    if manual_merge_map[kw] in phrase_rename_map:
        manual_merge_map[kw] = phrase_rename_map[manual_merge_map[kw]]
# Ensure old representative names map to new names in merge_map
for old_name, new_name in phrase_rename_map.items():
    if old_name not in merge_map:
        merge_map[old_name] = new_name

print(f'\n{"=" * 70}')
print(f'Step 1 — Merge-First Consolidation (per-topic TF-IDF):')
print(f'  Before:  {len(df_all)} phrases')
print(f'  Manual-group merged: {sum(1 for kw in merge_map if kw in manual_merge_map)} phrases')
print(f'  TF-IDF auto-merged:  {len(tfidf_merge_map)} phrases')
print(f'  Force-merged (low sim):      {len(tfidf_force_merged)} phrases (none discarded)')
print(f'  After:   {len(df_step1)} phrases (consolidated representatives)')
print(f'{"=" * 70}')

Starting: 399 phrases total
  Rare (count < 3): 251
  Survivors (count >= 3): 148

Manual synonym groups: 16
Rare phrases matched by manual groups: 229
Rare phrases needing TF-IDF matching: 21

--- TF-IDF per-topic matching ---
  [DDIC] 1 rare → matched within 23 survivors (auto=1, force=0)
  [VEC ] 9 rare → matched within 33 survivors (auto=0, force=9)
  [STL ] 4 rare → matched within 21 survivors (auto=1, force=3)
  [SCH ] 1 rare → matched within 9 survivors (auto=0, force=1)
  [CSTD] 5 rare → matched within 28 survivors (auto=1, force=4)
  [AU  ] 0 rare to match
  [ICL ] 1 rare → matched within 29 survivors (auto=0, force=1)

TF-IDF total auto-merged (sim >= 0.35): 3
TF-IDF total force-merged (sim < 0.35): 18
  (No phrases discarded — all rare merged into closest same-topic synonym)

Step 1 — Merge-First Consolidation (per-topic TF-IDF):
  Before:  399 phrases
  Manual-group merged: 333 phrases
  TF-IDF auto-merged:  21 phrases
  Force-merged (low sim):      18 phrases (none discard

In [5]:
# ---- Merge diagnostics ----
if tfidf_merge_map:
    print('--- TF-IDF auto-merged phrases ---')
    for kw, (target, sim) in sorted(tfidf_merge_map.items(), key=lambda x: -x[1][1]):
        tag = '' if sim >= SIMILARITY_THRESHOLD else ' [FORCE]'
        print(f'  "{kw}" -> "{target}" (sim={sim:.3f}){tag}')

if tfidf_force_merged:
    print(f'\n--- Force-merged phrases (low similarity) ({len(tfidf_force_merged)}) ---')
    for kw, best, sim in sorted(tfidf_force_merged, key=lambda x: -x[2]):
        print(f'  "{kw}" → "{best}" (sim={sim:.3f})')

if rare_group_reps:
    print(f'\n--- Rare-only group representatives ({len(rare_group_reps)}) ---')
    for kw in sorted(rare_group_reps):
        members = [m for m in manual_merge_map if manual_merge_map[m] == kw and m != kw]
        print(f'  "{kw}" <- {members}')

print(f'\ndf_step1: {len(df_step1)} phrases')
print(f'\nTop 30 by count:')
for i, (_, row) in enumerate(df_step1.head(30).iterrows()):
    print(f'  {i+1:2d}. "{row["keyword"]}" (count={row["count"]}, vt={row["n_vistypes"]})')

--- TF-IDF auto-merged phrases ---
  "no measurement/metric" -> "no measurement/metric/axis" (sim=0.857)
  "distinguishable/full color uses" -> "distinguishable/full/highly contrast color uses" (sim=0.597)
  "clustered data/info" -> "little data/info" (sim=0.384)
  "clear color uses" -> "distinguishable/full/highly contrast color uses" (sim=0.295) [FORCE]
  "no square information" -> "no clear indication" (sim=0.186) [FORCE]
  "y-axis meaning unclear" -> "unclear relation between words" (sim=0.151) [FORCE]
  "precision of aspect" -> "shape point of interest" (sim=0.151) [FORCE]
  "readable/distinguishable colors" -> "distinguishable color variety/arrangement/distribution" (sim=0.115) [FORCE]
  "no color" -> "color shading" (sim=0.095) [FORCE]
  "unable to remember" -> "easy to understand" (sim=0.042) [FORCE]
  "mismatched number-circle proportion" -> "shapes and lines" (sim=0.000) [FORCE]
  "exact value" -> "shapes and lines" (sim=0.000) [FORCE]
  "exact feature" -> "shapes and lines" 

## Step 1b: Criterion-Based Semantic Filtering (Revised)

Adapted from BeauVis Section 5.1, with **Criterion 5 revised** to match the original BeauVis wording:

| # | BeauVis Criterion | Our Adaptation |
|---|---|---|
| 1 | "Related to aesthetic pleasure" | **Related to visual complexity** — keep all (already filtered by source) |
| 2 | "Appeared ≥ 2 times" | Handled by Step 1 merge |
| 3 | "Usable in a rating scale" | **Clear directional connotation** — remove purely empty/artifact entries |
| 4 | "Easy to understand" | **Unambiguous wording** — remove data artifacts |
| 5 | "Clearly applies to a visual representation" | **REVISED: Keep domain-specific terms** — only exclude phrases with no visual dimension at all |
| 6 | "No opposite-pair redundancy" | **Remove antonym pairs** — keep higher-count member |

**Key change:** Criterion 5 no longer excludes single-VisType phrases. This preserves Schema/domain-specific
phrases that were previously lost.

In [6]:
# ---- Step 1b: Criterion-based semantic filtering (REVISED) ----

# Criterion 3 & 4: Remove data artifacts / empty entries only
vague_or_nondirectional = [
    '-',  # data artifact (empty/missing entry)
]

# Criterion 5 REVISED: "Clearly applies to a visual representation"
# We keep domain-specific and single-VisType phrases.
# Only exclude phrases that describe NO visual property at all.
# (In practice, all our phrases come from visual complexity descriptions,
#  so nothing is excluded here beyond the artifacts above.)
non_visual = []  # placeholder — all phrases in our data describe visual properties

# Criterion 6: Antonym-pair dedup — keep higher-count member
antonym_lower = []
antonym_pairs = [
    ('familiar representation', 'unfamiliar concepts/patterns'),
    ('organized/structured', 'looks random/messy/lack structure'),
    ('high contrast', 'lack of color contrast / hard to distinguish colors'),
    ('good color contrast/separation/segmentation', 'low color contrast/separation/segmentation'),
    ('clearer indication', 'no clear indication'),
]
for pos, neg in antonym_pairs:
    pos_row = df_step1[df_step1['keyword'] == pos]
    neg_row = df_step1[df_step1['keyword'] == neg]
    if len(pos_row) > 0 and len(neg_row) > 0:
        if pos_row.iloc[0]['count'] >= neg_row.iloc[0]['count']:
            antonym_lower.append(neg)
        else:
            antonym_lower.append(pos)

# Combine exclusions (NO single-VisType exclusion)
exclude_crit34 = set(vague_or_nondirectional)
exclude_crit5 = set(non_visual)
exclude_crit6 = set(antonym_lower)
all_excluded = exclude_crit34 | exclude_crit5 | exclude_crit6

df_step1b = df_step1[~df_step1['keyword'].isin(all_excluded)].copy().reset_index(drop=True)

print(f'Step 1b — Criterion-based semantic filtering (REVISED):')
print(f'  Before: {len(df_step1)} phrases')
print(f'  Criterion 3/4 (data artifacts):      removed {len(exclude_crit34)} phrases')
print(f'  Criterion 5 (non-visual):            removed {len(exclude_crit5 - exclude_crit34)} additional phrases')
print(f'  Criterion 6 (antonym lower-count):   removed {len(exclude_crit6 - exclude_crit34 - exclude_crit5)} additional phrases')
print(f'  After:  {len(df_step1b)} phrases')

# NOTE: Single-VisType phrases are KEPT (unlike the original pipeline)
single_vt_count = len(df_step1b[df_step1b['n_vistypes'] <= 1])
print(f'\n  Single-VisType phrases retained: {single_vt_count}')

if antonym_lower:
    print(f'\n--- Removed by Criterion 6 (lower-count antonym) ---')
    for kw in sorted(antonym_lower):
        row = df_step1[df_step1['keyword'] == kw]
        if len(row) > 0:
            print(f'  "{kw}" (count={row.iloc[0]["count"]})')

Step 1b — Criterion-based semantic filtering (REVISED):
  Before: 50 phrases
  Criterion 3/4 (data artifacts):      removed 1 phrases
  Criterion 5 (non-visual):            removed 0 additional phrases
  Criterion 6 (antonym lower-count):   removed 0 additional phrases
  After:  50 phrases

  Single-VisType phrases retained: 2


## Step 2: Universality Scoring with Secondary Topic

$$\text{universality\_score} = \text{count} \times \text{n\_vistypes}$$

Each phrase is assigned a `primary_topic` (first listed) and `secondary_topic` (second listed, if any).
Phrases appearing frequently AND across many VisTypes score highest.

In [7]:
# ---- Step 2: Universality scoring + topic assignment ----
df_step2 = df_step1b.copy()
df_step2['universality_score'] = df_step2['count'] * df_step2['n_vistypes']
df_step2 = df_step2.sort_values('universality_score', ascending=False).reset_index(drop=True)

# Primary topic = the representative's ORIGINAL topic (from df_all, before merging expanded it)
# This preserves Schema's domain-specific phrases under Schema, not under
# alphabetically-first topics like "Aesthetics Uncertainty"
orig_topic_map = dict(zip(df_all['keyword'], df_all['topics_involved']))
# Map renamed phrases to their original topic assignments
for old_name, new_name in phrase_rename_map.items():
    if old_name in orig_topic_map and new_name not in orig_topic_map:
        orig_topic_map[new_name] = orig_topic_map[old_name]

# Topic overrides: fix group reps whose auto-selected representative
# inherited a different topic than the group's semantic home
topic_overrides = {
    'unclear meaning/confusing': 'Immediacy / Cognitive Load',
}
for kw, topic in topic_overrides.items():
    orig_topic_map[kw] = topic

def get_primary_topic(row):
    kw = row['keyword']
    # Use the original (pre-merge) topic assignment
    orig_topics = orig_topic_map.get(kw, '')
    if orig_topics:
        parts = [t.strip() for t in orig_topics.split(TOPIC_SEP)]
        return parts[0] if parts else 'Unknown'
    # Fallback to post-merge topics_involved
    topics = row['topics_involved']
    if pd.notna(topics) and topics:
        return topics.split(TOPIC_SEP)[0].strip()
    return 'Unknown'

df_step2['primary_topic'] = df_step2.apply(get_primary_topic, axis=1)

# Secondary topic = second topic in the MERGED topics_involved (shows cross-topic coverage)
def get_secondary_topic(row):
    topics = row['topics_involved']
    primary = row['primary_topic']
    if pd.isna(topics) or not topics:
        return ''
    parts = [t.strip() for t in topics.split(TOPIC_SEP)]
    # Return the first topic that isn't the primary
    for t in parts:
        if t != primary:
            return t
    return ''

df_step2['secondary_topic'] = df_step2.apply(get_secondary_topic, axis=1)

print(f'Step 2 — Universality scoring:')
print(f'  Phrases: {len(df_step2)}')
print(f'  Phrases with secondary_topic: {(df_step2["secondary_topic"] != "").sum()}')

# Verify Schema coverage
for topic in topic_names:
    n = len(df_step2[df_step2['primary_topic'] == topic])
    print(f'  [{topic_abbrev[topic]:4s}] {topic:45s}: {n:2d} phrases')

print(f'\nTop 40 by universality score:')
display_cols = ['keyword', 'count', 'n_vistypes', 'universality_score', 'primary_topic', 'secondary_topic']
print(df_step2[display_cols].head(40).to_string(index=True))

Step 2 — Universality scoring:
  Phrases: 50
  Phrases with secondary_topic: 17
  [DDIC] Data Density / Image Clutter                 :  6 phrases
  [VEC ] Visual Encoding Clarity                      : 15 phrases
  [STL ] Semantics / Text Legibility                  :  7 phrases
  [SCH ] Schema                                       :  3 phrases
  [CSTD] Color, Symbol, and Texture Details           : 12 phrases
  [AU  ] Aesthetics Uncertainty                       :  2 phrases
  [ICL ] Immediacy / Cognitive Load                   :  5 phrases

Top 40 by universality score:
                                                    keyword  count  n_vistypes  universality_score                       primary_topic                     secondary_topic
0                                    easy/hard to interpret    299           9                2691          Immediacy / Cognitive Load              Aesthetics Uncertainty
1   domain-specific concepts (e.g., chemical, biology, map)    177           9

## Step 3: Sub-Topic Taxonomy (Topic-by-Topic)

Instead of selecting a flat shortlist of top phrases, we define **named sub-topics** within each of the 7 topics. Each sub-topic is a higher-level concept that groups multiple consolidated phrases (and their underlying original keywords).

**Strategy:** Process one topic at a time, assigning every post-merge phrase to a semantically coherent sub-topic with:
- A **name** (conceptual label, not a representative phrase)
- A **description** (what aspect of visual complexity it captures)
- **Member phrases** (consolidated phrases from Step 1 that belong here)

This gives complete coverage of all ~400 original keywords while reducing them to ~21 interpretable sub-topics.

In [8]:
# ---- Step 3: Sub-Topic Taxonomy (topic-by-topic) ----
#
# Each sub-topic: name, description, list of consolidated phrases (from df_step2)
# The code then maps ALL original keywords through merge_map → sub-topic.

subtopic_taxonomy = {
    # ================================================================
    # DDIC — Data Density / Image Clutter
    # ================================================================
    'Data Density / Image Clutter': [
        {
            'name': 'Information Volume',
            'description': (
                'The perceived amount, richness, or depth of data content '
                'in the image — ranging from sparse/simple to dense/layered information.'
            ),
            'phrases': [
                'much/little data/info',
                'more parameters/variables',
            ],
        },
        {
            'name': 'Element Quantity',
            'description': (
                'The number of discrete graphical elements (points, lines, bars, '
                'shapes, subplots) present in the image.'
            ),
            'phrases': [
                'more charts/points/lines/shapes/elements',
                'scattered shapes (e.g., dots, squares, pictures)',
            ],
        },
        {
            'name': 'Visual Clutter & Overlap',
            'description': (
                'Spatial density, overlapping elements, layout congestion, '
                'and use (or lack) of whitespace.'
            ),
            'phrases': [
                'overlapping shapes/colors/lines',
                'more regions/space',
            ],
        },
    ],

    # ================================================================
    # VEC — Visual Encoding Clarity
    # ================================================================
    'Visual Encoding Clarity': [
        {
            'name': 'Graphical Forms & Primitives',
            'description': (
                'The variety, type, and physical attributes of shapes, lines, '
                'and mark types used to encode data — including size, thickness, '
                'curvature, and regularity of forms.'
            ),
            'phrases': [
                'shapes and lines',
                'shape variation',
                'shape size',
                'shape increase/decrease point',
                'random-like shapes',
                'simple visual elements',
                'several elements',
            ],
        },
        {
            'name': 'Position, Scale & Organization',
            'description': (
                'Spatial layout, alignment, ordering, scale consistency, '
                'and structural organization of visual encodings.'
            ),
            'phrases': [
                'scale difference',
                'shape position',
                'ordered',
                'less organized elements',
            ],
        },
        {
            'name': 'Encoding Interpretability',
            'description': (
                'Whether visual encodings convey clear, decodable meaning — '
                'including recognizability of marks, design-data correspondence, '
                'and ease of identifying data features.'
            ),
            'phrases': [
                'clear colors/shapes',
                'understand/read shapes',
                'shape point of interest',
                'design-data relationship',
            ],
        },
    ],

    # ================================================================
    # STL — Semantics / Text Legibility
    # ================================================================
    'Semantics / Text Legibility': [
        {
            'name': 'Annotations & Labels',
            'description': (
                'Presence, clarity, and sufficiency of titles, axis labels, '
                'legends, captions, and measurement references.'
            ),
            'phrases': [
                'labels/axes/legends',
                'more labels/context',
                'clear text/scale/information',
            ],
        },
        {
            'name': 'Text Volume & Content',
            'description': (
                'The quantity of text, numbers, or contextual descriptions '
                'present in the image.'
            ),
            'phrases': [
                'amount of words/context/numbers',
                'little/less text/numbers/context',
            ],
        },
        {
            'name': 'Typography & Readability',
            'description': (
                'Font size, rotation, structure, spatial arrangement, '
                'and visual clarity of text elements.'
            ),
            'phrases': [
                'word rotation/small font size',
                'clear text arrangement/density',
            ],
        },
    ],

    # ================================================================
    # SCH — Schema
    # ================================================================
    'Schema': [
        {
            'name': 'Domain Familiarity',
            'description': (
                'Whether the image requires specialized domain knowledge '
                '(e.g., chemistry, biology, geography) or uses unfamiliar '
                'or specialized representational conventions.'
            ),
            'phrases': [
                'domain-specific concepts (e.g., chemical, biology, map)',
            ],
        },
        {
            'name': 'Dimensionality & Structure',
            'description': (
                'Spatial dimensionality (2D vs 3D), layout organization, '
                'and the structural framework of the visualization.'
            ),
            'phrases': [
                '2D/3D',
            ],
        },
        {
            'name': 'Abstraction Level',
            'description': (
                'Use of non-representational, abstract visual forms '
                'that lack familiar real-world referents.'
            ),
            'phrases': [
                'abstract shapes or colors',
            ],
        },
    ],

    # ================================================================
    # CSTD — Color, Symbol, and Texture Details
    # ================================================================
    'Color, Symbol, and Texture Details': [
        {
            'name': 'Color Palette & Contrast',
            'description': (
                'The range, variety, and arrangement of colors used — '
                'including gradients, hues, saturation, color schemes — '
                'and whether colors are visually distinguishable and provide '
                'clear separation, contrast, and background effects.'
            ),
            'phrases': [
                'color variety/shading',
                'two colors',
                'monochrome',
                'bright colors',
                'inadequate/few coloring',
                'color contrast/clarity',
                'distinguishable/full/highly contrast color uses',
                'distinguishable color variety/arrangement/distribution',
                'soft/layered/overlapping colors',
                'blurry colors/images',
                'white/black background / other background',
            ],
        },
        {
            'name': 'Symbols & Texture',
            'description': (
                'Non-color graphical markers, textures, icons, and '
                'patterned visual elements used alongside or instead of color.'
            ),
            'phrases': [
                'symbols',
            ],
        },
    ],

    # ================================================================
    # AU — Aesthetics Uncertainty
    # ================================================================
    'Aesthetics Uncertainty': [
        {
            'name': 'Visual Disorganization',
            'description': (
                'Whether the image feels random, messy, distracting, '
                'inconsistent, overwhelming, or lacking coherent structure.'
            ),
            'phrases': [
                'distracting/confusing/unclear',
            ],
        },
        {
            'name': 'Perceptual Ambiguity',
            'description': (
                'Uncertainty in perceiving specific visual attributes '
                'like color and contrast, leading to aesthetic confusion.'
            ),
            'phrases': [
                'unclear colors/contrast',
            ],
        },
    ],

    # ================================================================
    # ICL — Immediacy / Cognitive Load
    # ================================================================
    'Immediacy / Cognitive Load': [
        {
            'name': 'Interpretive Difficulty',
            'description': (
                'Overall subjective ease or difficulty of interpreting, '
                'reading, understanding, or differentiating the visualization.'
            ),
            'phrases': [
                'easy/hard to interpret',
                'more difficult to focus/follow',
            ],
        },
        {
            'name': 'Semantic Clarity',
            'description': (
                'Whether the visualization conveys an unambiguous, understandable '
                'message about what the data represents.'
            ),
            'phrases': [
                'unclear meaning/confusing',
            ],
        },
        {
            'name': 'Processing Time & Effort',
            'description': (
                'The time, cognitive effort, or specialized knowledge needed '
                'to process and analyze the visualization.'
            ),
            'phrases': [
                'take longer to interpret',
                'read/analyze many figures/items',
            ],
        },
    ],
}

# ---- Build phrase → sub-topic mapping and validate ----
phrase_to_subtopic = {}   # consolidated phrase → (topic, subtopic_name)
subtopic_records = []     # for output DataFrame

step2_kws = set(df_step2['keyword'])
unmapped_phrases = set(step2_kws)
missing_phrases = []

for topic in topic_names:
    abbrev = topic_abbrev[topic]
    subtopics = subtopic_taxonomy[topic]
    topic_phrases = df_step2[df_step2['primary_topic'] == topic]

    for st in subtopics:
        st_name = st['name']
        st_desc = st['description']
        matched_rows = []
        for phrase in st['phrases']:
            if phrase in step2_kws:
                row = df_step2[df_step2['keyword'] == phrase].iloc[0]
                matched_rows.append(row)
                phrase_to_subtopic[phrase] = (topic, st_name)
                unmapped_phrases.discard(phrase)
            else:
                missing_phrases.append((topic, st_name, phrase))

        # Aggregate stats
        total_count = sum(r['count'] for r in matched_rows)
        total_score = sum(r['universality_score'] for r in matched_rows)
        max_vt = max((r['n_vistypes'] for r in matched_rows), default=0)
        n_orig = 0
        orig_keywords = []
        for r in matched_rows:
            kw = r['keyword']
            merged_in = [k for k, v in merge_map.items() if v == kw]
            orig_keywords.extend(merged_in)
            orig_keywords.append(kw)
            n_orig += 1 + len(merged_in)

        subtopic_records.append({
            'topic': topic,
            'topic_abbrev': abbrev,
            'subtopic': st_name,
            'description': st_desc,
            'n_phrases': len(matched_rows),
            'total_count': total_count,
            'total_score': total_score,
            'max_vistypes': max_vt,
            'n_orig_keywords': n_orig,
            'phrases': '; '.join(r['keyword'] for r in matched_rows),
            'orig_keywords_sample': '; '.join(sorted(set(orig_keywords))[:15]),
        })

# ---- Diagnostics ----
n_subtopics = len(subtopic_records)
n_mapped = len(phrase_to_subtopic)

if unmapped_phrases:
    print(f'⚠ Unmapped phrases ({len(unmapped_phrases)}):')
    for p in sorted(unmapped_phrases):
        row = df_step2[df_step2['keyword'] == p]
        if len(row):
            r = row.iloc[0]
            print(f'    "{p}" (topic={r["primary_topic"]}, count={r["count"]}, score={r["universality_score"]})')
if missing_phrases:
    print(f'⚠ Phrases in taxonomy but not in df_step2 ({len(missing_phrases)}):')
    for topic, st, p in missing_phrases:
        print(f'    [{topic_abbrev[topic]}] {st}: "{p}"')

# Build summary DataFrame
df_subtopics = pd.DataFrame(subtopic_records)

# ---- Render as Markdown table ----
from IPython.display import display, Markdown

md_lines = ['| Topic | Sub-topic | Description | Phrases | Count | Score | Orig KWs |',
            '|-------|-----------|-------------|--------:|------:|------:|---------:|']
for topic in topic_names:
    abbrev = topic_abbrev[topic]
    topic_rows = df_subtopics[df_subtopics['topic'] == topic]
    first_in_topic = True
    for _, sr in topic_rows.iterrows():
        topic_cell = f'**[{abbrev}]** {topic}' if first_in_topic else ''
        phrase_list = '<br>'.join(f'`{p.strip()}`' for p in sr['phrases'].split(';') if p.strip())
        md_lines.append(
            f'| {topic_cell} | **{sr["subtopic"]}** | {sr["description"]} '
            f'| {phrase_list} '
            f'| {sr["total_count"]} | {sr["total_score"]} | {sr["n_orig_keywords"]} |'
        )
        first_in_topic = False

display(Markdown('\n'.join(md_lines)))

# ---- Save phrase_shortlist.csv (matches table above) ----
import os

shortlist_rows = []
for topic in topic_names:
    topic_rows = df_subtopics[df_subtopics['topic'] == topic]
    for _, sr in topic_rows.iterrows():
        phrase_nl = '\n'.join(p.strip() for p in sr['phrases'].split(';') if p.strip())
        shortlist_rows.append({
            'Topic': topic,
            'SubTopic': sr['subtopic'],
            'Description': sr['description'],
            'Phrases': phrase_nl,
            'Count': sr['total_count'],
            'Score': sr['total_score'],
            'Orig KWs': sr['n_orig_keywords'],
        })

df_shortlist_out = pd.DataFrame(shortlist_rows)
shortlist_path = os.path.join(out_dir, 'phrase_shortlist.csv')
df_shortlist_out.to_csv(shortlist_path, index=False)
print(f'Saved: {shortlist_path}  ({len(df_shortlist_out)} sub-topics)')

| Topic | Sub-topic | Description | Phrases | Count | Score | Orig KWs |
|-------|-----------|-------------|--------:|------:|------:|---------:|
| **[DDIC]** Data Density / Image Clutter | **Information Volume** | The perceived amount, richness, or depth of data content in the image — ranging from sparse/simple to dense/layered information. | `much/little data/info`<br>`more parameters/variables` | 164 | 1431 | 20 |
|  | **Element Quantity** | The number of discrete graphical elements (points, lines, bars, shapes, subplots) present in the image. | `more charts/points/lines/shapes/elements`<br>`scattered shapes (e.g., dots, squares, pictures)` | 143 | 1263 | 19 |
|  | **Visual Clutter & Overlap** | Spatial density, overlapping elements, layout congestion, and use (or lack) of whitespace. | `overlapping shapes/colors/lines`<br>`more regions/space` | 69 | 603 | 14 |
| **[VEC]** Visual Encoding Clarity | **Graphical Forms & Primitives** | The variety, type, and physical attributes of shapes, lines, and mark types used to encode data — including size, thickness, curvature, and regularity of forms. | `shapes and lines`<br>`shape variation`<br>`shape size`<br>`shape increase/decrease point`<br>`random-like shapes`<br>`simple visual elements`<br>`several elements` | 166 | 1349 | 60 |
|  | **Position, Scale & Organization** | Spatial layout, alignment, ordering, scale consistency, and structural organization of visual encodings. | `scale difference`<br>`shape position`<br>`ordered`<br>`less organized elements` | 13 | 43 | 4 |
|  | **Encoding Interpretability** | Whether visual encodings convey clear, decodable meaning — including recognizability of marks, design-data correspondence, and ease of identifying data features. | `clear colors/shapes`<br>`understand/read shapes`<br>`shape point of interest`<br>`design-data relationship` | 19 | 82 | 5 |
| **[STL]** Semantics / Text Legibility | **Annotations & Labels** | Presence, clarity, and sufficiency of titles, axis labels, legends, captions, and measurement references. | `labels/axes/legends`<br>`more labels/context`<br>`clear text/scale/information` | 142 | 1230 | 31 |
|  | **Text Volume & Content** | The quantity of text, numbers, or contextual descriptions present in the image. | `amount of words/context/numbers`<br>`little/less text/numbers/context` | 90 | 786 | 20 |
|  | **Typography & Readability** | Font size, rotation, structure, spatial arrangement, and visual clarity of text elements. | `word rotation/small font size`<br>`clear text arrangement/density` | 57 | 441 | 16 |
| **[SCH]** Schema | **Domain Familiarity** | Whether the image requires specialized domain knowledge (e.g., chemistry, biology, geography) or uses unfamiliar or specialized representational conventions. | `domain-specific concepts (e.g., chemical, biology, map)` | 177 | 1593 | 29 |
|  | **Dimensionality & Structure** | Spatial dimensionality (2D vs 3D), layout organization, and the structural framework of the visualization. | `2D/3D` | 41 | 369 | 20 |
|  | **Abstraction Level** | Use of non-representational, abstract visual forms that lack familiar real-world referents. | `abstract shapes or colors` | 5 | 25 | 1 |
| **[CSTD]** Color, Symbol, and Texture Details | **Color Palette & Contrast** | The range, variety, and arrangement of colors used — including gradients, hues, saturation, color schemes — and whether colors are visually distinguishable and provide clear separation, contrast, and background effects. | `color variety/shading`<br>`two colors`<br>`monochrome`<br>`bright colors`<br>`inadequate/few coloring`<br>`color contrast/clarity`<br>`distinguishable/full/highly contrast color uses`<br>`distinguishable color variety/arrangement/distribution`<br>`soft/layered/overlapping colors`<br>`blurry colors/images`<br>`white/black background / other background` | 302 | 2458 | 60 |
|  | **Symbols & Texture** | Non-color graphical markers, textures, icons, and patterned visual elements used alongside or instead of color. | `symbols` | 12 | 84 | 8 |
| **[AU]** Aesthetics Uncertainty | **Visual Disorganization** | Whether the image feels random, messy, distracting, inconsistent, overwhelming, or lacking coherent structure. | `distracting/confusing/unclear` | 66 | 594 | 19 |
|  | **Perceptual Ambiguity** | Uncertainty in perceiving specific visual attributes like color and contrast, leading to aesthetic confusion. | `unclear colors/contrast` | 8 | 48 | 1 |
| **[ICL]** Immediacy / Cognitive Load | **Interpretive Difficulty** | Overall subjective ease or difficulty of interpreting, reading, understanding, or differentiating the visualization. | `easy/hard to interpret`<br>`more difficult to focus/follow` | 302 | 2700 | 47 |
|  | **Semantic Clarity** | Whether the visualization conveys an unambiguous, understandable message about what the data represents. | `unclear meaning/confusing` | 83 | 747 | 20 |
|  | **Processing Time & Effort** | The time, cognitive effort, or specialized knowledge needed to process and analyze the visualization. | `take longer to interpret`<br>`read/analyze many figures/items` | 68 | 580 | 11 |

Saved: d:\Coding\Copilot\comment_post_processing\phrase_reduction_v2\phrase_shortlist.csv  (19 sub-topics)


## Step 4: Save Sub-Topic Taxonomy and Reduction Tracking Table

Save the sub-topic taxonomy and a full tracking table showing what happened to every original phrase.

In [9]:
# ---- Save sub-topic taxonomy ----
csv_path = os.path.join(out_dir, 'subtopic_taxonomy.csv')
df_subtopics.to_csv(csv_path, index=False)
print(f'Saved sub-topic taxonomy ({len(df_subtopics)} sub-topics) to: {csv_path}')
df_subtopics[['topic_abbrev', 'subtopic', 'description', 'n_phrases', 'total_count', 'total_score', 'n_orig_keywords']]

Saved sub-topic taxonomy (19 sub-topics) to: d:\Coding\Copilot\comment_post_processing\phrase_reduction_v2\subtopic_taxonomy.csv


,topic_abbrev,subtopic,description,n_phrases,total_count,total_score,n_orig_keywords
0,DDIC,Information Volume,"The perceived amount, richness, or depth of da...",2,164,1431,20
1,DDIC,Element Quantity,The number of discrete graphical elements (poi...,2,143,1263,19
2,DDIC,Visual Clutter & Overlap,"Spatial density, overlapping elements, layout ...",2,69,603,14
3,VEC,Graphical Forms & Primitives,"The variety, type, and physical attributes of ...",7,166,1349,60
4,VEC,"Position, Scale & Organization","Spatial layout, alignment, ordering, scale con...",4,13,43,4
5,VEC,Encoding Interpretability,"Whether visual encodings convey clear, decodab...",4,19,82,5
6,STL,Annotations & Labels,"Presence, clarity, and sufficiency of titles, ...",3,142,1230,31
7,STL,Text Volume & Content,"The quantity of text, numbers, or contextual d...",2,90,786,20
8,STL,Typography & Readability,"Font size, rotation, structure, spatial arrang...",2,57,441,16
9,SCH,Domain Familiarity,Whether the image requires specialized domain ...,1,177,1593,29


In [10]:
# ---- Build reduction tracking table ----
step1_kws = set(df_step1['keyword'])
step1b_kws = set(df_step1b['keyword'])
mapped_kws = set(phrase_to_subtopic.keys())  # phrases assigned to a sub-topic

tracking_rows = []
for _, row in df_all.iterrows():
    kw = row['keyword']
    count = row['count']
    n_vt = row['n_vistypes']
    topics = row['topics_involved']

    # Phase 1: Was it merged in Step 1?
    if kw in merge_map:
        rep = merge_map[kw]
        source = 'manual group' if kw in manual_merge_map else 'TF-IDF'
        sim_info = ''
        if kw in tfidf_merge_map:
            sim_info = f' (sim={tfidf_merge_map[kw][1]:.3f})'
        # Find sub-topic of the representative
        st_info = phrase_to_subtopic.get(rep, ('', ''))
        tracking_rows.append({
            'keyword': kw, 'count': count, 'n_vistypes': n_vt, 'topics_involved': topics,
            'step1_merge': f'MERGED ({source}){sim_info}',
            'step1b_criteria': '-', 'step3_subtopic': st_info[1] if st_info[1] else '-',
            'removed_at': f'Step 1: merged into "{rep}"',
            'final_status': 'merged',
            'merged_into': rep
        })
        continue

    # Phase 2: Was it removed in Step 1b?
    if kw not in step1b_kws:
        reason = []
        if kw in exclude_crit34:
            reason.append('Crit3/4: data artifact')
        if kw in exclude_crit5:
            reason.append('Crit5: non-visual')
        if kw in exclude_crit6:
            reason.append('Crit6: antonym lower-count')
        reason_str = '; '.join(reason) if reason else 'criterion filter'
        tracking_rows.append({
            'keyword': kw, 'count': count, 'n_vistypes': n_vt, 'topics_involved': topics,
            'step1_merge': 'kept (rep)',
            'step1b_criteria': 'REMOVED', 'step3_subtopic': '-',
            'removed_at': f'Step 1b: {reason_str}',
            'final_status': 'removed',
            'merged_into': ''
        })
        continue

    # Phase 3: Was it assigned to a sub-topic in Step 3?
    if kw in mapped_kws:
        st_info = phrase_to_subtopic[kw]
        tracking_rows.append({
            'keyword': kw, 'count': count, 'n_vistypes': n_vt, 'topics_involved': topics,
            'step1_merge': 'kept (rep)',
            'step1b_criteria': 'kept', 'step3_subtopic': st_info[1],
            'removed_at': '',
            'final_status': 'ASSIGNED',
            'merged_into': ''
        })
    else:
        tracking_rows.append({
            'keyword': kw, 'count': count, 'n_vistypes': n_vt, 'topics_involved': topics,
            'step1_merge': 'kept (rep)',
            'step1b_criteria': 'kept', 'step3_subtopic': 'UNMAPPED',
            'removed_at': 'Step 3: not assigned to any sub-topic',
            'final_status': 'unmapped',
            'merged_into': ''
        })

df_tracking = pd.DataFrame(tracking_rows)
df_tracking = df_tracking.sort_values(['final_status', 'count'], ascending=[True, False]).reset_index(drop=True)

tracking_path = os.path.join(out_dir, 'phrase_reduction_tracking.csv')
df_tracking.to_csv(tracking_path, index=False)

# Summary
print('PHRASE REDUCTION TRACKING TABLE')
print('=' * 70)
status_counts = df_tracking['final_status'].value_counts()
for status in ['ASSIGNED', 'merged', 'unmapped', 'removed']:
    n = status_counts.get(status, 0)
    print(f'  {status:<20s}: {n:>4d}')
print(f'  {"TOTAL":<20s}: {len(df_tracking):>4d}')
print(f'\nSaved to: {tracking_path}')

PHRASE REDUCTION TRACKING TABLE
  ASSIGNED            :   45
  merged              :  354
  unmapped            :    0
  removed             :    0
  TOTAL               :  399

Saved to: d:\Coding\Copilot\comment_post_processing\phrase_reduction_v2\phrase_reduction_tracking.csv


In [11]:
# ---- Pipeline summary ----
print('=' * 70)
print('PHRASE REDUCTION PIPELINE SUMMARY (v2 — Sub-Topic Taxonomy)')
print('=' * 70)
n_manual_merged = sum(1 for kw in merge_map if kw in manual_merge_map)
n_tfidf_merged = len(tfidf_merge_map)
n_force_merged = len(tfidf_force_merged)
n_assigned = (df_tracking['final_status'] == 'ASSIGNED').sum()
n_unmapped = (df_tracking['final_status'] == 'unmapped').sum()
print(f'  Step 0  — Raw unique phrases:              {len(df_all):>4d}')
print(f'  Step 1  — After merge-first consolidation:  {len(df_step1):>4d}')
print(f'            (manual-group merged: {n_manual_merged}, TF-IDF merged: {n_tfidf_merged}, force-merged: {n_force_merged})')
print(f'  Step 1b — After criterion filtering:        {len(df_step1b):>4d}  (-{len(df_step1) - len(df_step1b)})')
print(f'  Step 2  — Scored by universality:           {len(df_step2):>4d}  (same, just scored)')
print(f'  Step 3  — Sub-topic taxonomy:               {len(df_subtopics):>4d} sub-topics')
print(f'            Phrases assigned: {n_assigned}, Unmapped: {n_unmapped}')
print(f'\n  Overall: {len(df_all)} keywords → {len(df_subtopics)} sub-topics across 7 topics')
print('=' * 70)
print(f'\n  Saved: {csv_path}')
print(f'  Saved: {tracking_path}')

PHRASE REDUCTION PIPELINE SUMMARY (v2 — Sub-Topic Taxonomy)
  Step 0  — Raw unique phrases:               399
  Step 1  — After merge-first consolidation:    50
            (manual-group merged: 333, TF-IDF merged: 21, force-merged: 18)
  Step 1b — After criterion filtering:          50  (-0)
  Step 2  — Scored by universality:             50  (same, just scored)
  Step 3  — Sub-topic taxonomy:                 19 sub-topics
            Phrases assigned: 45, Unmapped: 0

  Overall: 399 keywords → 19 sub-topics across 7 topics

  Saved: d:\Coding\Copilot\comment_post_processing\phrase_reduction_v2\subtopic_taxonomy.csv
  Saved: d:\Coding\Copilot\comment_post_processing\phrase_reduction_v2\phrase_reduction_tracking.csv


## Per-Image Table: imageName, VisType, Topics, finalPhrases

Build a table mapping each image to its VisType, topics, and the final shortlisted phrases (after merge + reduction).


In [37]:
import json

# ====================================================================
# 3-SHEET CHAIN: curated originals → HumanCuration → final subtopics
# ====================================================================
# Sheet 1455332435  : term mapping (curated original → HumanCuration)
# Sheet 1540814304  : image pairs  (image names, VisTypes, curated original phrases per topic)
# Sheet 517808299  : already loaded as df_keyword_counts (HumanCuration phrases by topic)
# Chain: image → curated original → HumanCuration → pipeline shortlist

# == Step 1: Load term mapping (sheet 1455332435) ==
df_term_map = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?gid=1455332435&format=csv'
)

# Build forward mapping:  (topic, curated_original) → HumanCuration phrase
orig_to_hc = {}
for topic in topic_names:
    col_orig = topic
    col_hc   = f'{topic} (HumanCuration)'
    if col_orig in df_term_map.columns and col_hc in df_term_map.columns:
        for _, row in df_term_map.iterrows():
            orig = row.get(col_orig)
            hc   = row.get(col_hc)
            if pd.notna(orig) and pd.notna(hc):
                orig_to_hc[(topic, str(orig).strip())] = str(hc).strip()

print(f'Step 1 — Term mapping: {len(orig_to_hc)} (topic, original) → HumanCuration entries')
del df_term_map

# == Step 2: Load image pairs (sheet 1540814304) ==
df_pairs = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?gid=1540814304&format=csv'
)
df_pairs = df_pairs[df_pairs['ToRemove'].isna()].copy()
print(f'Step 2 — Image pairs: {len(df_pairs)} valid rows')

vt_name_map = {
    'schematic': 'Schematic', 'Schematic': 'Schematic',
    'Points': 'Point', 'Node-link and area': 'Node-link', 'Table': 'Table',
    'Point;Line': 'Point', 'Color/Pattern': 'Cont.-ColorPatn', 'GUI': 'GUI',
}

# Topic-key normalization (some JSON keys may use " - " instead of " / ")
_topic_norm = {}
for t in topic_names:
    _topic_norm[t] = t
    _topic_norm[t.replace(' / ', ' - ')] = t

def _strip_sentiment(phrase):
    """Remove (+)/(-) prefix from curated phrases."""
    p = phrase.strip()
    if p.startswith('(+)') or p.startswith('(-)'):
        p = p[3:].strip()
    return p

def _parse_curated_json(cell_value):
    """Parse {topic: [phrases]} JSON from CuratePhrases column."""
    if not isinstance(cell_value, str) or cell_value.strip() == '':
        return {}
    try:
        d = json.loads(cell_value)
        result = {}
        for topic_key, phrases in d.items():
            topic = _topic_norm.get(topic_key, topic_key)
            if isinstance(phrases, list):
                result.setdefault(topic, [])
                for p in phrases:
                    cleaned = _strip_sentiment(str(p))
                    if cleaned:
                        result[topic].append(cleaned)
        return result
    except (json.JSONDecodeError, ValueError):
        return {}

# Build per-image data from pair rows
image_data = {}   # imageName → {'VisType': str, 'curated': {topic: set(phrases)}}

for _, row in df_pairs.iterrows():
    for img_col, vt_col, cp_col in [
        ('moreComplexImageName', 'VisTypesMoreComplexImageType', 'CuratePhrasesMoreComplex'),
        ('lessComplexImageName', 'VisTypesLessComplexImageType', 'CuratePhrasesLessComplex'),
    ]:
        img = row[img_col]
        vt  = row[vt_col]
        if pd.isna(img) or pd.isna(vt):
            continue
        vt_clean = vt_name_map.get(vt, vt)

        if img not in image_data:
            image_data[img] = {'VisType': vt_clean, 'curated': defaultdict(set)}

        curated = _parse_curated_json(row.get(cp_col, ''))
        for topic, phrases in curated.items():
            for p in phrases:
                image_data[img]['curated'][topic].add(p)

all_imgs = set(image_data.keys())
print(f'Unique images from pairs: {len(all_imgs)}')
del df_pairs

# == Step 3: Build phrase-to-final mapping ==
shortlist_kws = set(phrase_to_subtopic.keys())
phrase_to_final = {}
# Shortlist maps to itself
for kw in shortlist_kws:
    phrase_to_final[kw] = kw
# Merged phrases map to their representative (already renamed)
for raw, rep in merge_map.items():
    if rep in shortlist_kws:
        phrase_to_final[raw] = rep
# Pre-rename names map to post-rename names (HumanCuration uses originals)
for old_name, new_name in phrase_rename_map.items():
    if new_name in shortlist_kws:
        phrase_to_final[old_name] = new_name

print(f'Phrase-to-final mapping: {len(phrase_to_final)} entries → {len(shortlist_kws)} final phrases')

# == Step 4: Chain per-image: curated original → HumanCuration → final ==
img_rows = []
topic_cols = topic_names   # reuse downstream

for img_name, data in image_data.items():
    rec = {'ImageName': img_name, 'VisType': data['VisType']}

    all_hc       = []      # ordered list of HumanCuration phrases across topics
    all_final    = set()

    for topic in topic_names:
        hc_for_topic = set()
        for orig_phrase in data['curated'].get(topic, set()):
            hc = orig_to_hc.get((topic, orig_phrase))
            if hc:
                hc_for_topic.add(hc)
                if hc in phrase_to_final:
                    all_final.add(phrase_to_final[hc])
        # Per-topic column: semicolon-separated HumanCuration phrases
        rec[topic] = '; '.join(sorted(hc_for_topic)) if hc_for_topic else ''
        all_hc.extend(sorted(hc_for_topic))

    # Derive topics from final phrases' subtopic→topic assignments (not source JSON keys)
    matched_tops = []
    for fp in all_final:
        if fp in phrase_to_subtopic:
            t, _ = phrase_to_subtopic[fp]
            if t not in matched_tops:
                matched_tops.append(t)
    # Reorder to match topic_names order
    matched_tops = [t for t in topic_names if t in matched_tops]

    rec['_matched_topics'] = '; '.join(matched_tops)
    rec['_hc_all']         = '; '.join(dict.fromkeys(all_hc))   # dedup preserving order
    rec['_final_phrases']  = '; '.join(sorted(all_final))
    img_rows.append(rec)

df_images = pd.DataFrame(img_rows)
df_images['VisType_clean'] = df_images['VisType']
# Keep ALL VisTypes in data (including non-standard: Schematic, GUI, Table)
# Analysis cells filter to 9 standard VisTypes as needed

# df_per_image: only images with at least one matched final phrase
df_per_image = df_images[df_images['_final_phrases'] != ''][
    ['ImageName', 'VisType', '_matched_topics', '_hc_all', '_final_phrases']
].rename(columns={
    'ImageName': 'imageName',
    '_matched_topics': 'Topics',
    '_hc_all': 'humanCuratedPhrases',
    '_final_phrases': 'finalPhrases',
}).reset_index(drop=True)

print(f'\nResults (all VisTypes):')
print(f'  Total images : {len(df_images)}')
print(f'  With HC match: {(df_images["_hc_all"] != "").sum()}')
print(f'  With final   : {len(df_per_image)}')

# Per-VisType breakdown (9 standard)
print(f'\n  9 standard VisTypes:')
for vt in VisTypes:
    n_total = (df_images['VisType'] == vt).sum()
    n_match = (df_per_image['VisType'] == vt).sum()
    print(f'    {vt:18s}: {n_match:3d} / {n_total:3d}')
# Non-standard VisTypes
other_vts = sorted(set(df_images['VisType']) - set(VisTypes))
if other_vts:
    print(f'\n  Non-standard VisTypes:')
    for vt in other_vts:
        n_total = (df_images['VisType'] == vt).sum()
        n_match = (df_per_image['VisType'] == vt).sum()
        print(f'    {vt:18s}: {n_match:3d} / {n_total:3d}')

per_image_path = os.path.join(out_dir, 'image_final_phrases.csv')
df_per_image.to_csv(per_image_path, index=False)
print(f'\nSaved: {per_image_path}')
df_per_image.head(10)

Step 1 — Term mapping: 1777 (topic, original) → HumanCuration entries
Step 2 — Image pairs: 685 valid rows
Unique images from pairs: 700
Phrase-to-final mapping: 405 entries → 50 final phrases

Results (all VisTypes):
  Total images : 700
  With HC match: 525
  With final   : 525

  9 standard VisTypes:
    Area              :  66 /  84
    Bar               :  51 /  78
    Cont.-ColorPatn   :  42 /  61
    Glyph             :  64 /  80
    Grid              :  65 /  79
    Line              :  48 /  70
    Node-link         :  67 /  85
    Point             :  58 /  78
    Text              :  52 /  68

  Non-standard VisTypes:
    Area and Text     :   1 /   1
    Bar and point     :   1 /   1
    GUI               :   0 /   1
    Schematic         :   9 /  13
    Table             :   1 /   1

Saved: d:\Coding\Copilot\comment_post_processing\phrase_reduction_v2\image_final_phrases.csv


,imageName,VisType,Topics,humanCuratedPhrases,finalPhrases
0,vis652.png,Point,Data Density / Image Clutter; Semantics / Text...,dense/cluttered layout; little data/info; more...,amount of words/context/numbers; color contras...
1,InfoVisJ.1933.13.png,Grid,Aesthetics Uncertainty; Immediacy / Cognitive ...,distracting/confusing/unclear; read/analyze ma...,distracting/confusing/unclear; read/analyze ma...
2,economist_daily_chart_521.png,Line,Schema; Aesthetics Uncertainty; Immediacy / Co...,"domain-specific concepts (e.g., chemical, biol...",distracting/confusing/unclear; domain-specific...
3,visMost786.png,Area,Immediacy / Cognitive Load,easier to interpret/read/understand,easy/hard to interpret
4,InfoVisJ.647.7(2).png,Text,Aesthetics Uncertainty,looks random/messy/lack structure,distracting/confusing/unclear
5,economist_daily_chart_393.png,Point,Data Density / Image Clutter; Visual Encoding ...,few points/lines/shapes/elements; little data/...,color contrast/clarity; domain-specific concep...
6,VisC.29.16.png,Line,Schema; Aesthetics Uncertainty,a specific technique; distracting/confusing/un...,distracting/confusing/unclear; domain-specific...
7,SciVisJ.980.7(2).png,Glyph,Immediacy / Cognitive Load,easy to interpret/read/understand,easy/hard to interpret
8,VASTJ.1698.5(4).png,Node-link,Data Density / Image Clutter; Visual Encoding ...,overlapping shapes/colors/lines; 2D/3D; inform...,2D/3D; distracting/confusing/unclear; domain-s...
9,VisC.295.3.png,Cont.-ColorPatn,Aesthetics Uncertainty; Immediacy / Cognitive ...,unclear colors/contrast; hard to focus/follow,easy/hard to interpret; unclear colors/contrast


In [39]:
# ---- Ultimate: image_phrase_word_mapping.csv ----
# Only images with matched final phrases (~520 expected)

import spacy, re
from nltk.stem import SnowballStemmer

nlp = spacy.load('en_core_web_sm')
stemmer = SnowballStemmer('english')

WHITELIST_TOKENS = {'2d', '3d', '2d/3d'}

def is_valid_token(token):
    text = token.text.lower()
    if text in WHITELIST_TOKENS:
        return True
    if token.is_stop or token.is_punct or token.is_space:
        return False
    if token.pos_ == 'NUM':
        return False
    if not re.match(r'^[a-z]+(-[a-z]+)*$', text):
        return False
    if len(text) <= 1:
        return False
    return True

# Build global stem-to-word mapping from all known phrases
stem_to_word = {}
all_phrases = list(phrase_to_subtopic.keys()) + list(merge_map.keys())
for ph in all_phrases:
    doc = nlp(ph)
    for token in doc:
        if is_valid_token(token):
            word = token.lemma_.lower()
            stem = stemmer.stem(word)
            if stem not in stem_to_word or len(word) < len(stem_to_word[stem]):
                stem_to_word[stem] = word

def extract_words(phrase_str):
    """Extract unique representative words from semicolon-separated phrases."""
    if not isinstance(phrase_str, str) or phrase_str.strip() == '':
        return ''
    phrases = [p.strip() for p in phrase_str.split(';') if p.strip()]
    stems_seen = set()
    words = []
    for ph in phrases:
        doc = nlp(ph)
        for token in doc:
            if is_valid_token(token):
                word = token.lemma_.lower()
                stem = stemmer.stem(word)
                if stem not in stems_seen:
                    stems_seen.add(stem)
                    words.append(stem_to_word.get(stem, word))
    return '; '.join(words)

# ---- Image URL base ----
IMAGE_URL_BASE = 'https://raw.githubusercontent.com/c109363/ExperimentImage/main/AllDataResize/'

# ---- Build mapping from df_per_image (only images with matched final phrases) ----
df_per_image = pd.read_csv(os.path.join(out_dir, 'image_final_phrases.csv'))
df_mapping = df_per_image.copy()
df_mapping['imageURL'] = IMAGE_URL_BASE + df_mapping['imageName']

# Derive SubTopics from humanCuratedPhrases (Topics already set from df_per_image)
def derive_subtopics(row):
    """Derive ordered SubTopics from humanCuratedPhrases."""
    hc = row.get('humanCuratedPhrases', '')
    if not isinstance(hc, str) or not hc.strip():
        return ''
    ordered_subtopics = []
    seen_subtopics = set()
    for p in hc.split(';'):
        p = p.strip()
        # HC phrase → final phrase (via phrase_to_final) → subtopic
        final = phrase_to_final.get(p)
        if final and final in phrase_to_subtopic:
            _, st = phrase_to_subtopic[final]
            if st not in seen_subtopics:
                ordered_subtopics.append(st)
                seen_subtopics.add(st)
    return '; '.join(ordered_subtopics)

df_mapping['SubTopics'] = df_mapping.apply(derive_subtopics, axis=1)

# Reorder columns
df_mapping = df_mapping[['imageName', 'imageURL', 'VisType', 'Topics',
                          'humanCuratedPhrases', 'SubTopics']]

# Save
mapping_path = os.path.join(out_dir, 'image_phrase_word_mapping.csv')
df_mapping.to_csv(mapping_path, index=False)
print(f'Saved: {mapping_path}  ({len(df_mapping)} images)')
print(f'Columns: {df_mapping.columns.tolist()}')

# Per-VisType summary
for vt in VisTypes:
    n = (df_mapping['VisType'] == vt).sum()
    if n > 0:
        print(f'  {vt:18s}: {n:3d} images')
# Non-standard VisTypes
other_vts = sorted(set(df_mapping['VisType']) - set(VisTypes))
for vt in other_vts:
    n = (df_mapping['VisType'] == vt).sum()
    if n > 0:
        print(f'  {vt:18s}: {n:3d} images (non-standard)')

df_mapping.head(5)

Saved: d:\Coding\Copilot\comment_post_processing\phrase_reduction_v2\image_phrase_word_mapping.csv  (525 images)
Columns: ['imageName', 'imageURL', 'VisType', 'Topics', 'humanCuratedPhrases', 'SubTopics']
  Area              :  66 images
  Bar               :  51 images
  Cont.-ColorPatn   :  42 images
  Glyph             :  64 images
  Grid              :  65 images
  Line              :  48 images
  Node-link         :  67 images
  Point             :  58 images
  Text              :  52 images
  Area and Text     :   1 images (non-standard)
  Bar and point     :   1 images (non-standard)
  Schematic         :   9 images (non-standard)
  Table             :   1 images (non-standard)


,imageName,imageURL,VisType,Topics,humanCuratedPhrases,SubTopics
0,vis652.png,https://raw.githubusercontent.com/c109363/Expe...,Point,Data Density / Image Clutter; Semantics / Text...,dense/cluttered layout; little data/info; more...,Visual Clutter & Overlap; Information Volume; ...
1,InfoVisJ.1933.13.png,https://raw.githubusercontent.com/c109363/Expe...,Grid,Aesthetics Uncertainty; Immediacy / Cognitive ...,distracting/confusing/unclear; read/analyze ma...,Visual Disorganization; Processing Time & Effort
2,economist_daily_chart_521.png,https://raw.githubusercontent.com/c109363/Expe...,Line,Schema; Aesthetics Uncertainty; Immediacy / Co...,"domain-specific concepts (e.g., chemical, biol...",Domain Familiarity; Visual Disorganization; In...
3,visMost786.png,https://raw.githubusercontent.com/c109363/Expe...,Area,Immediacy / Cognitive Load,easier to interpret/read/understand,Interpretive Difficulty
4,InfoVisJ.647.7(2).png,https://raw.githubusercontent.com/c109363/Expe...,Text,Aesthetics Uncertainty,looks random/messy/lack structure,Visual Disorganization


In [40]:
# ---- Align row order with canonical image list from sheet 483858074 ----
df_canonical = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?gid=483858074&format=csv'
)
print(f'Canonical sheet columns: {df_canonical.columns.tolist()}')
print(f'Canonical sheet rows: {len(df_canonical)}')

# Use imageName column (or first column as fallback)
img_col = 'imageName' if 'imageName' in df_canonical.columns else df_canonical.columns[0]
canonical_order = df_canonical[img_col].dropna().unique().tolist()
if img_col != 'imageName':
    df_canonical = df_canonical.rename(columns={img_col: 'imageName'})
print(f'Canonical image order: {len(canonical_order)} unique images')

# Keep only images that exist in the canonical sheet
canonical_set = set(canonical_order)
n_before = len(df_mapping)
df_mapping = df_mapping[df_mapping['imageName'].isin(canonical_set)].copy()
n_matched = len(df_mapping)

# Left join: keep all canonical images, fill unmatched with empty strings
df_canon_frame = pd.DataFrame({'imageName': canonical_order})
df_mapping = df_canon_frame.merge(df_mapping, on='imageName', how='left')
# Fill NaN for images in canonical sheet but not in our pipeline
for col in ['imageURL', 'VisType', 'Topics', 'humanCuratedPhrases', 'SubTopics']:
    if col in df_mapping.columns:
        df_mapping[col] = df_mapping[col].fillna('')

# Reorder columns
df_mapping = df_mapping[['imageName', 'imageURL', 'VisType', 'Topics',
                          'humanCuratedPhrases', 'SubTopics']]

print(f'Left join: {n_before} pipeline → {n_matched} matched canonical, {len(df_mapping)} total (all canonical kept)')

# Show unmatched canonical images (in sheet but no pipeline data)
unmatched = df_mapping[df_mapping['humanCuratedPhrases'] == '']['imageName'].tolist()
if unmatched:
    print(f'\n⚠ Unmatched canonical images ({len(unmatched)}):')
    for img in unmatched:
        print(f'    {img}')
else:
    print('All canonical images matched.')

# Re-save with canonical order
df_mapping.to_csv(mapping_path, index=False)
print(f'Re-saved (canonical order): {mapping_path}  ({len(df_mapping)} images)')
df_mapping.head(5)

Canonical sheet columns: ['image', 'imageName', 'imageURL', 'VisType', 'NormalizedVC', 'context', 'rawUserComments', 'originalPhrases', 'humanCuratedPhrases', 'originalSentiments', 'UniqueTopics', 'UniqueTopicsCount', 'UniqueSubTopics', 'UniqueSubTopicsCount', 'originalStems', 'originalStemPOS', 'objectWords', 'actionWords']
Canonical sheet rows: 520
Canonical image order: 520 unique images
Left join: 525 pipeline → 520 matched canonical, 520 total (all canonical kept)
All canonical images matched.
Re-saved (canonical order): d:\Coding\Copilot\comment_post_processing\phrase_reduction_v2\image_phrase_word_mapping.csv  (520 images)


,imageName,imageURL,VisType,Topics,humanCuratedPhrases,SubTopics
0,VisJ.1431.7.png,https://raw.githubusercontent.com/c109363/Expe...,Line,Visual Encoding Clarity; Immediacy / Cognitive...,shape increase/decrease point; unclear meaning...,Graphical Forms & Primitives; Semantic Clarity
1,InfoVisJ.2412.7.png,https://raw.githubusercontent.com/c109363/Expe...,Schematic,"Color, Symbol, and Texture Details",symbols,Symbols & Texture
2,SciVisJ.995.5.png,https://raw.githubusercontent.com/c109363/Expe...,Bar,Semantics / Text Legibility; Schema; Immediacy...,infomration representation; lack of/not enough...,Domain Familiarity; Annotations & Labels; Proc...
3,VisC.503.6.png,https://raw.githubusercontent.com/c109363/Expe...,Bar,Schema; Immediacy / Cognitive Load,"a specific technique, e.g., bar, pie, circle; ...",Domain Familiarity; Interpretive Difficulty
4,VisC.199.5.png,https://raw.githubusercontent.com/c109363/Expe...,Line,Immediacy / Cognitive Load,easy to interpret/read/understand; more interp...,Interpretive Difficulty
